# DS 5007 Natural Language Processing, Assignment 2
## Neural NLP Pipeline: a continuation of the BBC Urdu NLP pipeline

**Student:** i25-8023 | **Section:** DS-A

**GitHub repository:** https://github.com/ValeenaAfzal/i25-8023-NLP-Assignment2

Everything below is implemented from scratch in PyTorch (no pretrained models, no Gensim, no HuggingFace,
and no `nn.Transformer` / `nn.MultiheadAttention` / `nn.TransformerEncoder`). The inputs are the three files
produced in Assignment 1: `cleaned.txt` (sentence-per-line, normalised, lemmatised and stemmed), `raw.txt`
(untouched article bodies) and `Metadata.json` (titles, urls, dates).

| Part | Content |
|---|---|
| 0 | setup, data loading, topic labelling of the 250 articles (used by 1.1, 2 and 3) |
| 1 | TF-IDF, PPMI, t-SNE, Skip-gram Word2Vec, analogies, four-condition comparison |
| 2 | silver POS / NER annotation, BiLSTM tagger (softmax for POS, CRF + Viterbi for NER), evaluation, ablations |
| 3 | Transformer encoder topic classifier, attention heatmaps, BiLSTM vs Transformer comparison |

In [ ]:
# imports, seeds, paths and a helper that makes Urdu text render correctly inside matplotlib figures
import os, sys, re, json, math, time, random, copy
from collections import Counter, defaultdict
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.manifold import TSNE
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import arabic_reshaper
from bidi.algorithm import get_display
from IPython.display import display

sys.path.insert(0, "src")
from a1_tools import (read_cleaned, read_raw, corpus_word_freq, UrduLemmatizer, UrduStemmer,
                      NUM_TOKEN, PUNCT_TOKENS, PROTECTED_WORDS)
import urdu_lexicon as LEX

SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(8)
for folder in ("embeddings", "models", "data", "figures"):
    os.makedirs(folder, exist_ok=True)

plt.rcParams.update({"font.family": ["Segoe UI", "Tahoma", "Arial", "DejaVu Sans"], "figure.dpi": 100,
                     "axes.titlesize": 11, "axes.labelsize": 10})
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 250)


def ur(text):
    """reshape + bidi so that Urdu strings are joined and right-to-left inside matplotlib"""
    return get_display(arabic_reshaper.reshape(str(text)))


def savefig(name):
    plt.savefig(os.path.join("figures", name), dpi=150, bbox_inches="tight")


print("torch", torch.__version__, "| device:", DEVICE, "| threads:", torch.get_num_threads())

In [ ]:
# loading the three Assignment 1 files: cleaned.txt (main corpus), raw.txt (ablation baseline), Metadata.json
clean_docs = read_cleaned("cleaned.txt")
raw_docs = read_raw("raw.txt")
with open("Metadata.json", encoding="utf-8") as f:
    metadata = {m["article_number"]: m for m in json.load(f)}
ARTICLE_IDS = sorted(clean_docs)
assert ARTICLE_IDS == sorted(raw_docs) == sorted(metadata)

summary = pd.DataFrame({
    "file": ["cleaned.txt", "raw.txt"],
    "articles": [len(clean_docs), len(raw_docs)],
    "sentences": [sum(map(len, clean_docs.values())), sum(map(len, raw_docs.values()))],
    "tokens": [sum(len(s) for d in clean_docs.values() for s in d), sum(len(s) for d in raw_docs.values() for s in d)],
    "types": [len({t for d in clean_docs.values() for s in d for t in s}), len({t for d in raw_docs.values() for s in d for t in s})],
})
display(summary)
print("example cleaned sentence:", " ".join(clean_docs[1][0]))
print("example raw sentence    :", " ".join(raw_docs[1][0]))

### Topic labels for the 250 articles (Section 6.1, needed already in 1.1 and Part 2)
`Metadata.json` has no topic field, so every article is assigned to one of the five assignment categories by a
transparent keyword score. The indicative keywords from the handout (election, government, cricket, inflation,
UN, hospital, ...) are expanded with their Urdu equivalents. Score = keyword hits per 1,000 body tokens plus a
bonus of 15 for every keyword in the title (titles are short and very informative). Articles that hit no
keyword at all are human-interest stories and fall back to *Health & Society*.

In [ ]:
# keyword based assignment of every article to one of the five categories
TOPIC_KEYWORDS = {
    "Politics": """انتخاب الیکشن حکومت وزیر پارلیمان اسمبلی سیاسی سیاست جماعت پارٹی اپوزیشن سینیٹ ووٹ عدالت جج سپریم
                   آئین قانون عمران شریف تحریک_انصاف مسلم_لیگ پیپلز_پارٹی بلاول نواز شہباز مریم وزارت اعظم وزیراعلی گورنر کابینہ""",
    "Sports": """کرکٹ میچ ٹیم کھلاڑی سکور رنز وکٹ اننگز ٹیسٹ_میچ کپتان بیٹنگ بولنگ بولر فٹبال گول ٹورنامنٹ سیریز فائنل
                 ورلڈ کپ کھیل اولمپک ہاکی ٹینس میسی ارجنٹینا بلے گیند پی_سی_بی ٹی_ٹوئنٹی ون_ڈے""",
    "Economy": """مہنگائی تجارت بینک جی_ڈی_پی بجٹ معیشت معاشی اقتصادی روپے ڈالر قیمت تیل برآمد درآمد سرمایہ_کاری کمپنی
                  منڈی مارکیٹ قرض آئی_ایم_ایف ٹیکس شرح_سود کاروبار صنعت بیرل پیٹرول""",
    "International": """اقوام_متحدہ معاہدہ خارجہ دو_طرفہ تنازع جنگ ایران اسرائیل امریکہ امریکی چین روس یوکرین غزہ حوثی یمن
                        سعودی افغانستان طالبان سفارتی سفیر عالمی حملہ میزائل فوج فضائی سرحد مذاکرات ٹرمپ نیٹو انڈیا""",
    "Health & Society": """ہسپتال بیماری ویکسین سیلاب تعلیم صحت ڈاکٹر مریض علاج سکول طالب یونیورسٹی خاتون بچہ خاندان شادی
                           فلم اداکار گانا موسیقی سماجی معاشرہ پولیس قتل ماحول بارش موسم زلزلہ وبا دوا کہانی مذہب مرض امراض
                           کینسر نرس طبی حمل""",
}
TOPIC_KEYWORDS = {c: [k.replace("_", " ") for k in v.split()] for c, v in TOPIC_KEYWORDS.items()}
CATEGORIES = list(TOPIC_KEYWORDS)
CAT2ID = {c: i for i, c in enumerate(CATEGORIES)}


def topic_scores(num):
    body = [t for s in clean_docs[num] for t in s]
    joined = " " + " ".join(body) + " "
    title = metadata[num]["title"]
    return {c: 1000 * sum(joined.count(f" {k} ") for k in kws) / len(body) + 15 * sum(title.count(k) for k in kws)
            for c, kws in TOPIC_KEYWORDS.items()}


article_topic = {}
for num in ARTICLE_IDS:
    scores = topic_scores(num)
    best = max(scores, key=scores.get)
    article_topic[num] = best if scores[best] > 0 else "Health & Society"

with open("data/topic_labels.json", "w", encoding="utf-8") as f:
    json.dump({str(k): v for k, v in article_topic.items()}, f, ensure_ascii=False, indent=1)

topic_counts = Counter(article_topic.values())
display(pd.DataFrame({"articles": [topic_counts[c] for c in CATEGORIES]}, index=CATEGORIES).T)
for c in CATEGORIES:
    examples = [metadata[n]["title"][:70] for n in ARTICLE_IDS if article_topic[n] == c][:2]
    print(f"{c:17s} e.g. {examples}")

# Part 1: Word Embeddings
## 1.1 TF-IDF weighting
Vocabulary = the 10,000 most frequent tokens of `cleaned.txt`; every other token maps to `<UNK>`. Index 0 is
reserved for `<PAD>` (used by the neural models in Parts 2 and 3) and index 1 for `<UNK>`, so all matrices and
embedding tables share one `word2idx` of size 10,002. Documents are the 250 articles.

TF(w, d) = count(w, d) / |d| (length-normalised so long articles do not dominate the per-category averages),
IDF(w) = log[N / (1 + df(w))], TF-IDF = TF x IDF exactly as in the handout. Words that occur in (almost) every
article get IDF <= 0, which automatically pushes function words to the bottom.

In [ ]:
# vocabulary (10k most frequent tokens + <PAD>, <UNK>) and the raw term-document count matrix
VOCAB_SIZE = 10_000
PAD, UNK = "<PAD>", "<UNK>"
clean_counts = Counter(t for d in clean_docs.values() for s in d for t in s)
word2idx = {PAD: 0, UNK: 1}
for w, _ in clean_counts.most_common(VOCAB_SIZE):
    word2idx[w] = len(word2idx)
idx2word = {i: w for w, i in word2idx.items()}
V = len(word2idx)
with open("embeddings/word2idx.json", "w", encoding="utf-8") as f:
    json.dump(word2idx, f, ensure_ascii=False)


def encode(tokens, vocab=None):
    vocab = vocab or word2idx
    return [vocab.get(t, vocab[UNK]) for t in tokens]


covered = sum(c for w, c in clean_counts.items() if w in word2idx)
print(f"types in cleaned.txt: {len(clean_counts)} | vocabulary: {V} (incl. PAD, UNK) | "
      f"token coverage: {100 * covered / sum(clean_counts.values()):.2f}% | rarest kept word count: {clean_counts.most_common(VOCAB_SIZE)[-1][1]}")

N_DOCS = len(ARTICLE_IDS)
rows, cols = [], []
for r, num in enumerate(ARTICLE_IDS):
    ids = encode([t for s in clean_docs[num] for t in s])
    rows += [r] * len(ids)
    cols += ids
td_counts = sp.csr_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)), shape=(N_DOCS, V))
td_counts.sum_duplicates()
print("term-document matrix:", td_counts.shape, "| non-zeros:", td_counts.nnz,
      "| check: total count equals corpus tokens ->", int(td_counts.sum()) == sum(clean_counts.values()))
probe = ["پاکستان", "کرکٹ", "حکومت"]
display(pd.DataFrame(td_counts[:5][:, [word2idx[w] for w in probe]].toarray().astype(int), columns=probe,
                     index=[f"article {n}" for n in ARTICLE_IDS[:5]]))

In [ ]:
# TF-IDF = TF(w,d) x log(N / (1 + df(w))), saved as embeddings/tfidf_matrix.npy
doc_len = np.asarray(td_counts.sum(axis=1)).ravel()
tf = sp.diags(1.0 / doc_len) @ td_counts
df = np.asarray((td_counts > 0).sum(axis=0)).ravel()
idf = np.log(N_DOCS / (1.0 + df)).astype(np.float32)
tfidf = (tf @ sp.diags(idf)).toarray().astype(np.float32)
np.save("embeddings/tfidf_matrix.npy", tfidf)
print("tfidf_matrix.npy saved:", tfidf.shape, "| words with idf <= 0 (in >= 249 docs):",
      [idx2word[i] for i in np.where((idf <= 0) & (df > 0))[0]][:15])

In [ ]:
# top-10 most discriminative words per topic: mean TF-IDF inside the category minus mean TF-IDF in the other categories
# (a word must occur in at least max(3, 10%) of the category's articles, otherwise one long story dominates small classes)
SKIP_IDS = {word2idx[PAD], word2idx[UNK]} | {word2idx[t] for t in PUNCT_TOKENS | {NUM_TOKEN} if t in word2idx}
doc_topics = np.array([article_topic[n] for n in ARTICLE_IDS])
top_words = {}
for c in CATEGORIES:
    in_c = doc_topics == c
    inside, outside = tfidf[in_c].mean(axis=0), tfidf[~in_c].mean(axis=0)
    score = inside - outside
    spread = (td_counts[np.where(in_c)[0]] > 0).sum(axis=0).A1          # articles of c that contain the word
    score[spread < max(3, 0.1 * in_c.sum())] = -np.inf                 # ignore words living in a single story
    score[list(SKIP_IDS)] = -np.inf
    top_words[c] = [idx2word[i] for i in np.argsort(-score)[:10]]
display(pd.DataFrame(top_words, index=range(1, 11)))

DISC_TFIDF

## 1.2 Pointwise Mutual Information (PPMI)
Word-word co-occurrence counts with a symmetric window of k = 5 tokens on each side, inside sentence boundaries,
over the same 10,002-word vocabulary. PPMI(w1, w2) = max{0, log2[P(w1, w2) / (P(w1) P(w2))]} with all
probabilities estimated from the co-occurrence table.

In [ ]:
# co-occurrence counts with window k = 5, PPMI weighting, saved as embeddings/ppmi_matrix.npy
WINDOW = 5
centres, contexts = [], []
for d in clean_docs.values():
    for s in d:
        ids = np.array(encode(s))
        for off in range(1, WINDOW + 1):
            if len(ids) > off:
                centres += [ids[:-off], ids[off:]]
                contexts += [ids[off:], ids[:-off]]
centres, contexts = np.concatenate(centres), np.concatenate(contexts)
cooc = sp.csr_matrix((np.ones(len(centres), dtype=np.float64), (centres, contexts)), shape=(V, V))
cooc.sum_duplicates()


def ppmi_from_counts(counts):
    counts = counts.tocoo()
    total = counts.sum()
    row_p = np.asarray(counts.sum(axis=1)).ravel() / total
    col_p = np.asarray(counts.sum(axis=0)).ravel() / total
    pmi = np.log2((counts.data / total) / (row_p[counts.row] * col_p[counts.col]))
    keep = pmi > 0
    return sp.csr_matrix((pmi[keep].astype(np.float32), (counts.row[keep], counts.col[keep])), shape=counts.shape)


ppmi = ppmi_from_counts(cooc)
np.save("embeddings/ppmi_matrix.npy", ppmi.toarray())
print(f"co-occurrence pairs: {len(centres):,} | non-zero cells: {cooc.nnz:,} | positive PPMI cells: {ppmi.nnz:,} "
      f"({100 * ppmi.nnz / V ** 2:.2f}% dense) | symmetric: {abs(ppmi - ppmi.T).max() < 1e-4}")
print("ppmi_matrix.npy saved:", ppmi.shape, "float32")

In [ ]:
# cosine nearest neighbours helpers (work for dense embeddings and for sparse PPMI rows)
def l2_normalise(m):
    if sp.issparse(m):
        norms = np.sqrt(np.asarray(m.multiply(m).sum(axis=1)).ravel())
        return sp.diags(1.0 / np.maximum(norms, 1e-9)) @ m
    return m / np.maximum(np.linalg.norm(m, axis=1, keepdims=True), 1e-9)


def all_similarities(unit, idx):
    v = unit[idx]
    sims = unit @ (v.T if sp.issparse(unit) else v)
    return np.asarray(sims.todense()).ravel() if sp.issparse(sims) else np.asarray(sims).ravel()


MIN_EVAL_COUNT = 5   # like word2vec's default min_count: words seen < 5 times are not offered as neighbours / answers


def candidate_mask(vocab, counts, min_count=MIN_EVAL_COUNT):
    allowed = np.zeros(len(vocab), dtype=bool)
    for w, i in vocab.items():
        allowed[i] = counts.get(w, 0) >= min_count and w not in (PAD, UNK) and w not in PUNCT_TOKENS and w != NUM_TOKEN
    return allowed


def nearest(unit, vocab, inv, word, allowed, k=5):
    sims = all_similarities(unit, vocab[word])
    sims[~allowed] = -np.inf
    sims[vocab[word]] = -np.inf
    top = np.argsort(-sims)[:k]
    return [(inv[i], float(sims[i])) for i in top]


def neighbour_table(unit, vocab, inv, words, allowed, k=5):
    return pd.DataFrame({w: [f"{n} ({s:.2f})" for n, s in nearest(unit, vocab, inv, w, allowed, k)] for w in words if w in vocab},
                        index=range(1, k + 1))


clean_allowed = candidate_mask(word2idx, clean_counts)
print(f"neighbour / answer candidates (count >= {MIN_EVAL_COUNT}): {clean_allowed.sum():,} of {V:,} vocabulary entries")
ppmi_unit = l2_normalise(ppmi)
PPMI_QUERIES = ["پاکستان", "حکومت", "عدالت", "فوج", "کرکٹ", "ایران", "ہسپتال", "تیل", "فلم", "پولیس", "تعلیم", "میزائل"]
display(neighbour_table(ppmi_unit, word2idx, idx2word, PPMI_QUERIES, clean_allowed, k=5))

### t-SNE of the 200 most frequent tokens
Punctuation, `<NUM>` and function words (the A1 stop list plus the closed-class lexicon) carry no topical
meaning and would all collapse into one blob, so the 200 most frequent *content* tokens are used. Their PPMI rows
are reduced to 50 dimensions with truncated SVD and then projected with t-SNE. Each token was assigned to one of nine semantic
categories by hand (topical fields plus verbs and general discourse words).

In [ ]:
# picking the 200 most frequent content tokens and their hand-assigned semantic category
STOP = PROTECTED_WORDS | LEX.POST | LEX.PRON | LEX.CONJ | LEX.DET | LEX.AUX | LEX.NUM_WORDS | LEX.LETTER_NAMES | PUNCT_TOKENS | {NUM_TOKEN}
SEMANTIC = {
    "Politics & Law": """حکومت وزیر عدالت قانون صدر فیصلہ بیان الزام سیاسی اعظم مقدمہ عمران پارلیمان انتخاب سربراہ حکام سرکار
                         وزارت کارروائی درخواست جج آئین رکن تحقیق اختیار سابق اسمبلی قرار عائد""",
    "Security & Conflict": """فوج حملہ جنگ دفاع پولیس حوثی کارروائی میزائل اسرائیل طیارہ فضائی سکیورٹی نشانہ قتل طالبان جہاز
                              خطرہ ہلاک ڈرون حملے تنازع اہلکار گرفتار""",
    "Geography & Nations": """پاکستان انڈیا ایران سعودی عرب امریکہ امریکی چین ملک افغانستان اسلام آباد پنجاب شہر یمن علاقہ دنیا
                              عالم انڈین انگلینڈ سپین ارجنٹینا مغرب بحر جنوب مرکز ریاست لاہور کراچی افغان""",
    "Sports & Entertainment": """ٹیم کرکٹ کھلاڑی میچ ٹیسٹ ورلڈ کپ رنز کھیل فلم اداکار وکٹ کپتان فائنل سیریز گول اننگز گیند میسی
                                 بال کامیاب شکست ریکارڈ کارکردگی کردار کہانی بیٹنگ""",
    "Economy & Media": """تیل قیمت کمپنی روپے لاکھ کار میڈیا سوشل ویڈیو ٹی پوسٹ صارف خبر رپورٹ لکھا کتاب استعمال نقصان""",
    "People & Society": """لوگ خاتون بچہ فرد بیٹا خاندان زندگی گھر شادی عمر عوام مرد انسان ڈاکٹر خان محمد علی احمد بھائی
                           قوم شخص تعلیم والد""",
    "Time": """وقت سال دن برس ماہ ستمبر اگست روز دہائی آغاز تاریخ گذشتہ حالیہ صدی مہینہ ہفتہ آج دوران سنہ""",
    "Verbs & light verbs": """کر کہا کرنے کرتے کہنا ہونے جا کرنا بتایا جانے کرتا کہتے بن آنے دیتے جات کرتی آیا کریں دینے دکھائی
                              آئے بنانے دیکھ رکھنے دیکھا ادا حاصل شامل جاری پیش شروع قائم فراہم پیدا تیار متاثر واقع ظاہر
                              بند سامنا یاد نظر""",
    "Descriptors & discourse": """طور بات بڑا دوسرا مزید نام نیا غیر کوشش تقریبا پہلا کام بار سامنے معاملہ طرح اہم حصہ مقام
                                  واقعہ واپس مکمل تعلق ادارہ عمل انداز نظام سوال بنیاد پورا آخر مشکل دور موقع دعوی جواب
                                  صلاحیت باعث قریب بے آف تنقید دار واضح مسئلہ دوبارہ بین یقین ضرور جگہ گفتگو سطح اثر اتنا
                                  مدد مسلسل معمول خیال موجود""",
}
SEMANTIC_OF = {}
for cat, words in SEMANTIC.items():
    for w in words.split():
        SEMANTIC_OF.setdefault(w, cat)

tsne_words = [w for w, _ in clean_counts.most_common() if w not in STOP and len(w) > 1][:200]
tsne_cats = [SEMANTIC_OF.get(w, "Other / general") for w in tsne_words]
print(Counter(tsne_cats))
print("words left in 'Other / general':", " ".join(w for w, c in zip(tsne_words, tsne_cats) if c == "Other / general"))

In [ ]:
# SVD (50 dims) of the PPMI rows followed by t-SNE, colour coded by semantic category
svd_input = ppmi[[word2idx[w] for w in tsne_words]]
reduced = TruncatedSVD(n_components=50, random_state=SEED).fit_transform(svd_input)
coords = TSNE(n_components=2, perplexity=25, init="pca", random_state=SEED, metric="cosine").fit_transform(reduced)

palette = dict(zip(list(SEMANTIC) + ["Other / general"],
                   ["#d62728", "#7f3c8d", "#1f77b4", "#2ca02c", "#ff7f0e", "#17becf", "#8c564b", "#bcbd22", "#e377c2", "#b0b0b0"]))
fig, ax = plt.subplots(figsize=(13, 9))
for cat, colour in palette.items():
    sel = [i for i, c in enumerate(tsne_cats) if c == cat]
    ax.scatter(coords[sel, 0], coords[sel, 1], c=colour, s=28 if cat != "Other / general" else 14, label=f"{cat} ({len(sel)})",
               alpha=0.9 if cat != "Other / general" else 0.5)
    for i in sel:
        if cat != "Other / general" or i < 60:
            ax.annotate(ur(tsne_words[i]), coords[i], fontsize=8, alpha=0.85, xytext=(2, 2), textcoords="offset points")
ax.set_title("t-SNE of PPMI vectors for the 200 most frequent content tokens (colour = semantic category)")
ax.set_xlabel("t-SNE dimension 1")
ax.set_ylabel("t-SNE dimension 2")
ax.legend(loc="best", fontsize=9, title="semantic category")
savefig("tsne_ppmi.png")
plt.show()

DISC_TSNE

## 2. Skip-gram Word2Vec
### 2.1 Implementation
* training stream: sentences of the corpus, punctuation and `<NUM>` removed, frequent words sub-sampled
  (Mikolov et al., keep probability sqrt(t/f) + t/f with t = 1e-4, redrawn every epoch);
* positive pairs: every (centre, context) pair inside a symmetric window of k = 5 tokens;
* noise distribution P_n(w) proportional to f(w)^(3/4), implemented as a 10^6-entry unigram **noise table**
  (as in the original C code), K = 10 noise words per positive pair;
* two separate embedding matrices, V (centre) and U (context), both |V| x d;
* loss L = -log sigma(u_o . v_c) - sum_k log sigma(-u_k . v_c), averaged over the batch;
* Adam, eta = 0.001, batch 1,024 pairs (>= 512), d = 100, 10 epochs (>= 5), loss logged every 250 steps
  (fused Adam kernel for speed on CPU).

In [ ]:
# training-stream builder, noise table and skip-gram pair generator
W2V_EPOCHS, W2V_BATCH, W2V_LR, NEG_K, SUBSAMPLE_T = 10, 1024, 1e-3, 10, 1e-4
NOISE_TABLE_SIZE = 1_000_000
NON_WORD_RE = re.compile(r"^[\W\d_]+$")


def training_stream(docs, vocab):
    """list of id-arrays (one per sentence) without punctuation / numbers / PAD"""
    sentences = []
    for d in docs.values():
        for s in d:
            ids = [vocab.get(t, vocab[UNK]) for t in s if t not in PUNCT_TOKENS and t != NUM_TOKEN and not NON_WORD_RE.match(t)]
            if len(ids) > 1:
                sentences.append(np.array(ids, dtype=np.int64))
    return sentences


def build_noise_table(sentences, vocab_size, power=0.75, size=NOISE_TABLE_SIZE):
    freq = np.bincount(np.concatenate(sentences), minlength=vocab_size).astype(np.float64)
    probs = freq ** power
    probs /= probs.sum()
    table = np.repeat(np.arange(vocab_size), np.round(probs * size).astype(np.int64))
    return torch.from_numpy(table), freq / freq.sum(), probs


def skipgram_pairs(sentences, word_prob, rng, window=WINDOW, t=SUBSAMPLE_T):
    keep_p = np.minimum(1.0, np.sqrt(t / np.maximum(word_prob, 1e-12)) + t / np.maximum(word_prob, 1e-12))
    kept = [s[rng.random(len(s)) < keep_p[s]] for s in sentences]
    kept = [s for s in kept if len(s) > 1]
    ids = np.concatenate(kept)
    sent_id = np.repeat(np.arange(len(kept)), [len(s) for s in kept])
    cs, xs = [], []
    for off in range(1, window + 1):
        same = sent_id[:-off] == sent_id[off:]
        a, b = ids[:-off][same], ids[off:][same]
        cs += [a, b]
        xs += [b, a]
    return np.concatenate(cs), np.concatenate(xs)


clean_stream = training_stream(clean_docs, word2idx)
noise_table, clean_word_prob, noise_probs = build_noise_table(clean_stream, V)
demo_c, demo_x = skipgram_pairs(clean_stream, clean_word_prob, np.random.default_rng(SEED))
print(f"training sentences: {len(clean_stream):,} | stream tokens: {sum(map(len, clean_stream)):,} | "
      f"pairs in one epoch after sub-sampling: {len(demo_c):,} | noise table: {len(noise_table):,} entries")
top_noise = np.argsort(-noise_probs)[:8]
display(pd.DataFrame({"word": [idx2word[i] for i in top_noise], "unigram P(w)": clean_word_prob[top_noise].round(4),
                      "noise P_n(w) ~ f^0.75": noise_probs[top_noise].round(4)}))
print("first skip-gram pairs of sentence 1:", [(idx2word[c], idx2word[x]) for c, x in zip(demo_c[:6], demo_x[:6])])

In [ ]:
# the skip-gram model with separate centre (V) and context (U) matrices and the negative-sampling BCE loss
class SkipGramNS(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.V = nn.Embedding(vocab_size, dim)   # centre vectors v_c
        self.U = nn.Embedding(vocab_size, dim)   # context vectors u_o
        nn.init.uniform_(self.V.weight, -0.5 / dim, 0.5 / dim)
        nn.init.uniform_(self.U.weight, -0.5 / dim, 0.5 / dim)

    def forward(self, centre, context, negatives):
        v_c = self.V(centre)                                   # (B, d)
        u_o = self.U(context)                                  # (B, d)
        u_k = self.U(negatives)                                # (B, K, d)
        pos = F.logsigmoid((u_o * v_c).sum(-1))                # log sigma(u_o . v_c)
        neg = F.logsigmoid(-torch.bmm(u_k, v_c.unsqueeze(2)).squeeze(2)).sum(-1)   # sum_k log sigma(-u_k . v_c)
        return -(pos + neg).mean()

    def averaged_embeddings(self):
        emb = 0.5 * (self.V.weight + self.U.weight).detach().cpu().numpy()
        emb[0] = 0.0                                           # PAD row
        return emb


def train_word2vec(sentences, vocab_size, dim=100, epochs=W2V_EPOCHS, batch=W2V_BATCH, lr=W2V_LR, k=NEG_K, log_every=250, tag=""):
    set_seed()
    table, word_prob, _ = build_noise_table(sentences, vocab_size)
    model = SkipGramNS(vocab_size, dim).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, fused=True)
    rng = np.random.default_rng(SEED)
    curve, running, step, t0 = [], [], 0, time.time()
    for epoch in range(1, epochs + 1):
        c, x = skipgram_pairs(sentences, word_prob, rng)
        perm = rng.permutation(len(c))
        c, x = torch.from_numpy(c[perm]), torch.from_numpy(x[perm])
        epoch_losses = []
        for i in range(0, len(c), batch):
            cb, xb = c[i:i + batch].to(DEVICE), x[i:i + batch].to(DEVICE)
            negs = table[torch.randint(len(table), (len(cb), k))].to(DEVICE)
            loss = model(cb, xb, negs)
            opt.zero_grad()
            loss.backward()
            opt.step()
            running.append(loss.item())
            epoch_losses.append(loss.item())
            step += 1
            if step % log_every == 0:
                curve.append((step, float(np.mean(running))))
                running = []
        print(f"[{tag}] epoch {epoch:2d} | pairs {len(c):,} | mean loss {np.mean(epoch_losses):.4f} | {time.time() - t0:.0f}s")
    return model, curve


w2v_c3, curve_c3 = train_word2vec(clean_stream, V, dim=100, tag="C3 cleaned d=100")
emb_c3 = w2v_c3.averaged_embeddings()
np.save("embeddings/embeddings_w2v.npy", emb_c3)
print("embeddings_w2v.npy saved:", emb_c3.shape)

In [ ]:
# training loss curve of the main skip-gram model (C3)
steps, losses = zip(*curve_c3)
plt.figure(figsize=(8, 3.8))
plt.plot(steps, losses, color="#1f77b4")
plt.xlabel(f"training step (batch = {W2V_BATCH} pairs)")
plt.ylabel("negative-sampling BCE loss (mean of 250 steps)")
plt.title("Skip-gram training loss on cleaned.txt (d=100, k=5, K=10, Adam 1e-3)")
plt.grid(alpha=0.3)
savefig("w2v_loss_c3.png")
plt.show()

### 2.2 Evaluation: nearest neighbours and analogies
Query words (romanised in the handout): Pakistan = پاکستان, Hukumat = حکومت, Adalat = عدالت, Maeeshat = معیشت,
Fauj = فوج, Sehat = صحت, Taleem = تعلیم, Aabadi = آبادی. Note that the A1 stemmer strips the final ی of most
آبادی occurrences (they became آباد and merge with اسلام آباد), so only 3 tokens of آبادی survive in
`cleaned.txt`; its neighbours are therefore expected to be noisy.

In [ ]:
# top-10 cosine neighbours of the eight required query words (C3 embeddings)
c3_unit = l2_normalise(emb_c3)
QUERY_WORDS = ["پاکستان", "حکومت", "عدالت", "معیشت", "فوج", "صحت", "تعلیم", "آبادی"]
print("corpus frequency of the query words:", {w: clean_counts[w] for w in QUERY_WORDS})
display(neighbour_table(c3_unit, word2idx, idx2word, QUERY_WORDS, clean_allowed, k=10))
print("note: آبادی itself occurs only 3 times; it is still used as a query, but it is never offered as a neighbour.")

In [ ]:
# analogy tests a : b :: c : ?  answered with v(b) - v(a) + v(c), top-3 candidates (a, b, c excluded, count >= 5)
ANALOGIES = [   # 20 tests written before evaluation; the handout asks for 10, all 20 are reported
    ("ایران", "تہران", "افغانستان", "کابل"),       # country : capital
    ("انڈیا", "دہلی", "روس", "ماسکو"),             # country : capital
    ("امریکہ", "واشنگٹن", "برطانیہ", "لندن"),       # country : capital
    ("سعودی", "ریاض", "ایران", "تہران"),           # country : capital
    ("چین", "بیجنگ", "روس", "ماسکو"),              # country : capital
    ("لاہور", "پنجاب", "پشاور", "خیبر"),            # city : province
    ("لاہور", "پنجاب", "کراچی", "سندھ"),            # city : province
    ("انڈیا", "مودی", "امریکہ", "ٹرمپ"),            # country : leader
    ("امریکہ", "ٹرمپ", "روس", "پوتن"),             # country : leader
    ("میسی", "ارجنٹینا", "بابر", "پاکستان"),        # player : national team
    ("کرکٹ", "وکٹ", "فٹبال", "گول"),               # sport : scoring unit
    ("بیٹنگ", "بلے", "بولنگ", "گیند"),              # action : equipment
    ("عدالت", "جج", "ہسپتال", "ڈاکٹر"),            # institution : professional
    ("فلم", "اداکار", "کرکٹ", "کھلاڑی"),            # field : performer
    ("پاکستان", "روپے", "امریکہ", "ڈالر"),          # country : currency
    ("مرد", "عورت", "بھائی", "بہن"),               # gender
    ("شوہر", "بیوی", "والد", "والدہ"),              # gender
    ("اداکار", "اداکارہ", "مرد", "عورت"),           # gender
    ("بھائی", "بہن", "والد", "والدہ"),              # gender
    ("ستمبر", "اگست", "جولائی", "جون"),             # month : previous month
]


def analogy(unit, vocab, inv, a, b, c, allowed, k=3):
    q = unit[vocab[b]] - unit[vocab[a]] + unit[vocab[c]]
    sims = np.asarray(unit @ (q / np.linalg.norm(q))).ravel()
    sims[~allowed] = -np.inf
    sims[[vocab[a], vocab[b], vocab[c]]] = -np.inf
    return [inv[i] for i in np.argsort(-sims)[:k]]


def run_analogies(unit, vocab, inv, tests, allowed):
    rows = []
    for a, b, c, gold in tests:
        if not all(w in vocab for w in (a, b, c, gold)):
            rows.append({"test": f"{a} : {b} :: {c} : ?", "expected": gold, "top-3": "(word missing)", "correct": False})
            continue
        top = analogy(unit, vocab, inv, a, b, c, allowed)
        rows.append({"test": f"{a} : {b} :: {c} : ?", "expected": gold, "top-3": " | ".join(top), "correct": gold in top})
    return pd.DataFrame(rows)


analogy_c3 = run_analogies(c3_unit, word2idx, idx2word, ANALOGIES, clean_allowed)
display(analogy_c3)
print(f"correct (gold in top-3): {analogy_c3['correct'].sum()} / {len(analogy_c3)}  (first 10: {analogy_c3['correct'][:10].sum()} / 10)")

DISC_W2V

### Four-condition comparison
* **C1** PPMI rows (10,002-dim sparse vectors) from 1.2;
* **C2** skip-gram trained on `raw.txt` (whitespace / punctuation tokenisation only, its own 10k vocabulary);
* **C3** skip-gram on `cleaned.txt`, d = 100 (the model above);
* **C4** C3 with d = 200.

All skip-gram runs use identical hyper-parameters and seeds. **MRR** is computed on 20 hand-labelled related word
pairs (a, b): every vocabulary word is ranked by cosine similarity to a (a itself, PAD and UNK excluded),
RR = 1 / rank(b) and MRR is the mean over the 20 pairs. As everywhere in this part, only words seen at least 5 times
are candidates. All 40 words exist in both the cleaned and the raw vocabulary.

In [ ]:
# training C2 (raw corpus) and C4 (d = 200) with exactly the same settings
raw_counts = Counter(t for d in raw_docs.values() for s in d for t in s)
raw_word2idx = {PAD: 0, UNK: 1}
for w, _ in raw_counts.most_common(VOCAB_SIZE):
    raw_word2idx[w] = len(raw_word2idx)
raw_idx2word = {i: w for w, i in raw_word2idx.items()}
raw_stream = training_stream(raw_docs, raw_word2idx)
print(f"raw corpus: {len(raw_counts):,} types, vocabulary {len(raw_word2idx):,}, stream tokens {sum(map(len, raw_stream)):,}")

w2v_c2, curve_c2 = train_word2vec(raw_stream, len(raw_word2idx), dim=100, tag="C2 raw d=100")
emb_c2 = w2v_c2.averaged_embeddings()
w2v_c4, curve_c4 = train_word2vec(clean_stream, V, dim=200, tag="C4 cleaned d=200")
emb_c4 = w2v_c4.averaged_embeddings()
np.save("embeddings/embeddings_w2v_raw_c2.npy", emb_c2)
np.save("embeddings/embeddings_w2v_d200_c4.npy", emb_c4)

plt.figure(figsize=(8, 3.8))
for name, curve in (("C2 raw, d=100", curve_c2), ("C3 cleaned, d=100", curve_c3), ("C4 cleaned, d=200", curve_c4)):
    s, l = zip(*curve)
    plt.plot(s, l, label=name)
plt.xlabel(f"training step (batch = {W2V_BATCH} pairs)")
plt.ylabel("negative-sampling BCE loss")
plt.title("Skip-gram training loss for conditions C2, C3 and C4")
plt.legend()
plt.grid(alpha=0.3)
savefig("w2v_loss_conditions.png")
plt.show()

In [ ]:
# 20 hand-labelled related word pairs and the MRR / top-5 comparison of C1-C4
MRR_PAIRS = [
    ("کرکٹ", "میچ"), ("کھلاڑی", "ٹیم"), ("وکٹ", "رنز"), ("ایران", "اسرائیل"), ("افغانستان", "طالبان"),
    ("حوثی", "یمن"), ("سعودی", "عرب"), ("عدالت", "جج"), ("ہسپتال", "ڈاکٹر"), ("فلم", "اداکار"),
    ("پولیس", "گرفتار"), ("لاہور", "پنجاب"), ("میزائل", "ڈرون"), ("تیل", "گیس"), ("روپے", "ڈالر"),
    ("وزیر", "اعظم"), ("فوج", "فضائیہ"), ("امریکہ", "واشنگٹن"), ("چین", "بیجنگ"), ("تعلیم", "سکول"),
]
COMPARE_QUERIES = ["پاکستان", "حکومت", "عدالت", "فوج", "تعلیم"]


def mrr(unit, vocab, pairs, allowed):
    rrs = []
    for a, b in pairs:
        sims = all_similarities(unit, vocab[a])
        sims[~allowed] = -np.inf
        sims[vocab[a]] = -np.inf
        rank = int((sims > sims[vocab[b]]).sum()) + 1
        rrs.append(1.0 / rank)
    return float(np.mean(rrs)), rrs


raw_allowed = candidate_mask(raw_word2idx, raw_counts)
CONDITIONS = {
    "C1 PPMI": (ppmi_unit, word2idx, idx2word, clean_allowed),
    "C2 SG raw d=100": (l2_normalise(emb_c2), raw_word2idx, raw_idx2word, raw_allowed),
    "C3 SG cleaned d=100": (c3_unit, word2idx, idx2word, clean_allowed),
    "C4 SG cleaned d=200": (l2_normalise(emb_c4), word2idx, idx2word, clean_allowed),
}
assert all(a in raw_word2idx and b in raw_word2idx and a in word2idx and b in word2idx for a, b in MRR_PAIRS)
cond_rows, rr_table = [], {}
for name, (unit, vocab, inv, allowed) in CONDITIONS.items():
    score, rrs = mrr(unit, vocab, MRR_PAIRS, allowed)
    rr_table[name] = rrs
    ana = run_analogies(unit, vocab, inv, ANALOGIES, allowed) if not sp.issparse(unit) else None
    cond_rows.append({"condition": name, "MRR (20 pairs)": round(score, 3),
                      "pairs with rank 1": sum(r == 1.0 for r in rrs), "pairs in top 10": sum(r >= 0.1 for r in rrs),
                      f"analogies correct /{len(ANALOGIES)}": int(ana["correct"].sum()) if ana is not None else "n/a"})
display(pd.DataFrame(cond_rows).set_index("condition"))
display(pd.DataFrame(rr_table, index=[f"{a} - {b}" for a, b in MRR_PAIRS]).round(3))

for name, (unit, vocab, inv, allowed) in CONDITIONS.items():
    print(f"\n{name}: top-5 neighbours")
    display(neighbour_table(unit, vocab, inv, COMPARE_QUERIES, allowed, k=5))

DISC_COND

# Part 2: Sequence Labeling, POS Tagging and NER
## 3. Dataset preparation
500 sentences (5 to 40 tokens) are drawn at random from `cleaned.txt`: 100 from each of the five topic
categories, which satisfies "at least 100 sentences from each of 3 distinct categories" and makes the
topic-stratified split straightforward. Annotation is **silver** (rule generated): a rule-based POS tagger built
on the Assignment 1 lemmatizer / stemmer and a hand-crafted lexicon, and a gazetteer + rule based NER tagger. The
BiLSTM then has to learn to reproduce and generalise these labels from context.

In [ ]:
# drawing 100 sentences per topic category (500 in total) from cleaned.txt
MIN_LEN, MAX_LEN, PER_TOPIC = 5, 40, 100
rng_sel = random.Random(SEED)
selected = []
for c in CATEGORIES:
    pool = [(num, j) for num in ARTICLE_IDS if article_topic[num] == c
            for j, s in enumerate(clean_docs[num]) if MIN_LEN <= len(s) <= MAX_LEN]
    for num, j in rng_sel.sample(pool, PER_TOPIC):
        selected.append({"article": num, "sent_idx": j, "topic": c, "tokens": clean_docs[num][j]})
print("selected sentences:", len(selected), "|", dict(Counter(x["topic"] for x in selected)),
      "| tokens:", sum(len(x["tokens"]) for x in selected))

In [ ]:
# lexicon sizes: open classes have 200+ entries each, verbs = hand-picked roots + generated inflections
lex_sizes = pd.DataFrame({
    "entries": [len(LEX.NOUN), len(LEX.VERB), len(LEX.ADJ), len(LEX.ADV), len(LEX.PRON), len(LEX.DET),
                len(LEX.CONJ), len(LEX.POST), len(LEX.NUM_WORDS), len(LEX.PUNC)],
    "how built": ["hand list", f"{len(LEX.VERB_ROOTS_LEX)} hand roots x paradigm + {len(LEX.VERB_IRREGULAR)} irregular forms",
                  "hand list", "hand list", "closed class", "closed class", "closed class", "closed class (A1 list + extra)",
                  "closed class", "closed class"]},
    index=["NOUN", "VERB", "ADJ", "ADV", "PRON", "DET", "CONJ", "POST", "NUM", "PUNC"])
display(lex_sizes)
print("sample VERB paradigm for root کھیل:", sorted(LEX._inflect("کھیل")))

In [ ]:
# rule-based POS tagger on top of the A1 lemmatizer / stemmer
# the handout says "12 tags" but lists these 11; the 11 listed tags are used everywhere
POS_TAGS = ["NOUN", "VERB", "ADJ", "ADV", "PRON", "DET", "CONJ", "POST", "NUM", "PUNC", "UNK"]
URDU_WORD_RE = re.compile(r"^[ء-غف-يٮ-ۓە]+$")


class RuleBasedPOSTagger:
    OPEN = (("NOUN", LEX.NOUN), ("VERB", LEX.VERB), ("ADJ", LEX.ADJ), ("ADV", LEX.ADV))

    def __init__(self, word_freq):
        self.lemmatizer = UrduLemmatizer(word_freq)
        self.stemmer = UrduStemmer(word_freq)

    def open_classes(self, word):
        """open-class readings of a word: direct lookup first, then its A1 lemma, then its A1 stem"""
        for form in (word, self.lemmatizer.lemmatize(word), self.stemmer.stem(word)):
            hits = [tag for tag, lex in self.OPEN if form in lex]
            if hits:
                return hits
        return []

    def looks_nominal(self, word):
        return word is not None and word not in LEX.POST and word not in LEX.AUX and (
            set(self.open_classes(word)) & {"NOUN", "ADJ"} or
            (URDU_WORD_RE.match(word) and not self.open_classes(word) and word not in PROTECTED_WORDS))

    def tag_token(self, i, toks):
        t = toks[i]
        prev = toks[i - 1] if i > 0 else None
        nxt = toks[i + 1] if i + 1 < len(toks) else None
        clause_start = prev is None or prev in LEX.PUNC or prev in LEX.CONJ
        if t in LEX.PUNC:
            return "PUNC"
        if t == NUM_TOKEN or t in LEX.NUM_WORDS:
            return "NUM"
        if t == UNK or not URDU_WORD_RE.match(t):
            return "UNK"
        if t in LEX.LETTER_NAMES and ((prev in LEX.LETTER_NAMES) or (nxt in LEX.LETTER_NAMES)):
            return "UNK"                                             # transliterated acronym letters (بی بی سی)
        if t == "میں":
            return "PRON" if (clause_start or nxt in {"نے", "بھی", "خود"}) else "POST"
        if t in {"لیے", "لئے"}:
            return "POST" if prev in {"کے", "کی", "اس"} else "VERB"
        if t in {"کی", "کے"} and (nxt is None or nxt in LEX.PUNC or nxt in {"ہے", "ہیں", "تھی", "تھے", "گئی", "گئے", "جا", "جاتی", "جاتے", "جائے", "گی", "گے"}):
            return "VERB"                                            # light verb کرنا: 'بات کی۔'
        if t == "کیا":
            return "PRON" if (prev is None or nxt == "؟") else "VERB"
        if t in LEX.DEMONSTRATIVES:
            return "DET" if self.looks_nominal(nxt) else "PRON"
        for tag, lex in (("POST", LEX.POST), ("CONJ", LEX.CONJ), ("PRON", LEX.PRON), ("DET", LEX.DET)):
            if t in lex:
                return tag
        if t in LEX.AUX or t in LEX.VERB_IRREGULAR:
            return "VERB"
        hits = self.open_classes(t)
        verbal_context = nxt in LEX.AUX or nxt in {"کر", "کرنا", "کرنے", "کرتے", "دیا", "لیا"}
        if len(hits) == 1:
            return hits[0]
        if len(hits) > 1:
            if "VERB" in hits and verbal_context:
                return "VERB"
            if "ADJ" in hits and "NOUN" in self.open_classes(nxt or ""):
                return "ADJ"
            for tag in ("NOUN", "ADJ", "ADV", "VERB"):
                if tag in hits:
                    return tag
        # suffix rules for words the lexicon does not know
        if t.endswith(LEX.VERB_SUFFIXES) and verbal_context:
            return "VERB"
        if t.endswith(LEX.ADJ_SUFFIXES):
            return "ADJ"
        if t.endswith("ی") and len(t) > 3 and "NOUN" in self.open_classes(nxt or ""):
            return "ADJ"                                             # nisba adjective: سیاسی جماعت
        if t in LEX.LETTER_NAMES:
            return "UNK"
        return "NOUN"                                                # Urdu default: open-class noun

    def tag(self, tokens):
        return [self.tag_token(i, tokens) for i in range(len(tokens))]


pos_tagger = RuleBasedPOSTagger(corpus_word_freq(clean_docs))
for x in selected:
    x["pos"] = pos_tagger.tag(x["tokens"])
demo = selected[0]
print(" ".join(f"{t}/{p}" for t, p in zip(demo["tokens"], demo["pos"])))
print("silver POS distribution:", Counter(p for x in selected for p in x["pos"]).most_common())

In [ ]:
# seed gazetteer (>= 50 Pakistani persons, >= 50 locations, >= 30 organisations) and the rule-based BIO NER tagger
NER_TAGS = ["O", "B-PER", "I-PER", "B-LOC", "I-LOC", "B-ORG", "I-ORG", "B-MISC", "I-MISC"]
GAZETTEER = {}
for etype, phrases in (("MISC", LEX.MISC), ("LOC", LEX.LOCATIONS), ("ORG", LEX.ORGANISATIONS),
                       ("PER", LEX.FOREIGN_PERSONS), ("PER", LEX.PAKISTANI_PERSONS)):
    for ph in phrases:
        GAZETTEER[ph] = etype                                        # later lists win on exact duplicates
MAX_PHRASE = max(map(len, GAZETTEER))
display(pd.DataFrame({"entries": [len(LEX.PAKISTANI_PERSONS), len(LEX.FOREIGN_PERSONS), len(LEX.LOCATIONS),
                                  len(LEX.ORGANISATIONS), len(LEX.MISC)]},
                     index=["Pakistani persons", "foreign persons", "locations", "organisations", "misc (events, languages, demonyms, platforms)"]))


def ner_tag(tokens):
    tags = ["O"] * len(tokens)
    i = 0
    while i < len(tokens):                                           # 1) longest gazetteer match
        for n in range(min(MAX_PHRASE, len(tokens) - i), 0, -1):
            etype = GAZETTEER.get(tuple(tokens[i:i + n]))
            if etype:
                tags[i:i + n] = [f"B-{etype}"] + [f"I-{etype}"] * (n - 1)
                i += n
                break
        else:
            i += 1
    name_parts = LEX.FIRST_NAMES | LEX.SURNAMES
    i = 0
    while i < len(tokens):                                           # 2) title + name / first name + surname rules
        start = None
        if tags[i] == "O" and tokens[i] in LEX.PERSON_TITLES and i + 1 < len(tokens) and tokens[i + 1] in name_parts and tags[i + 1] == "O":
            start = i + 1
        elif tags[i] == "O" and tokens[i] in LEX.FIRST_NAMES and i + 1 < len(tokens) and tags[i + 1] == "O" and (
                tokens[i + 1] in name_parts or tokens[i + 1] in {"نے", "کا", "کی", "کے", "کو"}):
            start = i
        if start is None:
            i += 1
            continue
        end = start + 1
        while end < len(tokens) and end - start < 3 and tokens[end] in name_parts and tags[end] == "O":
            end += 1
        tags[start:end] = ["B-PER"] + ["I-PER"] * (end - start - 1)
        i = end
    return tags


for x in selected:
    x["ner"] = ner_tag(x["tokens"])
ex = next(x for x in selected if sum(t != "O" for t in x["ner"]) >= 3)
print(" ".join(f"{t}/{n}" if n != "O" else t for t, n in zip(ex["tokens"], ex["ner"])))
entity_counts = Counter(t[2:] for x in selected for t in x["ner"] if t.startswith("B-"))
print("silver entities:", dict(entity_counts), "| sentences with >= 1 entity:", sum(any(t != "O" for t in x["ner"]) for x in selected))

In [ ]:
# 70 / 15 / 15 split stratified by topic, CoNLL files and the label distribution of both tasks
idx_all = np.arange(len(selected))
topics_all = [x["topic"] for x in selected]
idx_train, idx_rest = train_test_split(idx_all, test_size=0.30, stratify=topics_all, random_state=SEED)
idx_val, idx_test = train_test_split(idx_rest, test_size=0.50, stratify=[topics_all[i] for i in idx_rest], random_state=SEED)
SPLITS = {"train": [selected[i] for i in idx_train], "val": [selected[i] for i in idx_val], "test": [selected[i] for i in idx_test]}


def write_conll(path, items, field):
    with open(path, "w", encoding="utf-8") as f:
        for x in items:
            for tok, tag in zip(x["tokens"], x[field]):
                f.write(f"{tok}\t{tag}\n")
            f.write("\n")


for split, items in SPLITS.items():
    write_conll(f"data/pos_{split}.conll", items, "pos")
    write_conll(f"data/ner_{split}.conll", items, "ner")
print({k: len(v) for k, v in SPLITS.items()}, "| topic counts per split:",
      {k: dict(Counter(x["topic"] for x in v)) for k, v in SPLITS.items()})

pos_dist = pd.DataFrame({k: Counter(p for x in v for p in x["pos"]) for k, v in SPLITS.items()}).reindex(POS_TAGS).fillna(0).astype(int)
ner_dist = pd.DataFrame({k: Counter(p for x in v for p in x["ner"]) for k, v in SPLITS.items()}).reindex(NER_TAGS).fillna(0).astype(int)
display(pos_dist.T)
display(ner_dist.T)

## 4. BiLSTM sequence labeler
* embedding layer initialised from the C3 skip-gram vectors (`embeddings_w2v.npy`, shared `word2idx`), used
  **frozen** or **fine-tuned**;
* 2-layer bidirectional LSTM (hidden 128 per direction); the per-token representation is
  h_t = [forward h_t ; backward h_t] (256-dim);
* dropout p = 0.5 between the two LSTM layers (and on the LSTM output);
* **POS**: linear layer + cross-entropy; **NER**: linear emission layer + **CRF** with a learnable tag-transition
  matrix (plus start / end transitions), trained with the forward algorithm, decoded with **Viterbi**;
* variable lengths: sequences are packed (`pack_padded_sequence`), pad labels are -100 / masked out, so padding
  never reaches the loss (CE `ignore_index`, CRF mask);
* Adam (eta = 1e-3, weight decay 1e-4), batch 16, at most 60 epochs, early stopping on validation F1 with
  patience 5 (POS: macro-F1, NER: conlleval entity F1; an epoch with equal F1 but lower validation loss
  also counts as progress, because entity F1 stays at 0 for the first epochs); the best checkpoint is restored.

In [ ]:
# datasets, padding-aware batching and the conlleval-style entity scorer
POS2ID = {t: i for i, t in enumerate(POS_TAGS)}
NER2ID = {t: i for i, t in enumerate(NER_TAGS)}
IGNORE = -100


def make_dataset(items, field, tag2id):
    return [(torch.tensor(encode(x["tokens"])), torch.tensor([tag2id[t] for t in x[field]])) for x in items]


def batches(data, batch_size=16, shuffle=False, rng=None):
    order = list(range(len(data)))
    if shuffle:
        rng.shuffle(order)
    for i in range(0, len(order), batch_size):
        chunk = [data[j] for j in order[i:i + batch_size]]
        lengths = torch.tensor([len(x) for x, _ in chunk])
        ids = nn.utils.rnn.pad_sequence([x for x, _ in chunk], batch_first=True, padding_value=0)
        tags = nn.utils.rnn.pad_sequence([y for _, y in chunk], batch_first=True, padding_value=IGNORE)
        yield ids, tags, lengths


def get_chunks(seq):
    """entity spans (type, start, end) of a BIO sequence, following conlleval's start/end-of-chunk rules"""
    chunks, start, ctype = [], None, None
    for i, tag in enumerate(list(seq) + ["O"]):
        prefix, etype = (tag.split("-", 1) + [None])[:2] if tag != "O" else ("O", None)
        ends = start is not None and (prefix in ("B", "O") or etype != ctype)
        if ends:
            chunks.append((ctype, start, i - 1))
            start, ctype = None, None
        if prefix == "B" or (prefix == "I" and start is None):
            start, ctype = i, etype
    return chunks


def conlleval(gold_seqs, pred_seqs, verbose=False):
    """entity-level precision / recall / F1 per type and overall (same counting as the conlleval perl script)"""
    stats = defaultdict(lambda: [0, 0, 0])                           # correct, guessed, gold
    tokens = correct_tokens = 0
    for g, p in zip(gold_seqs, pred_seqs):
        gc, pc = set(get_chunks(g)), set(get_chunks(p))
        for c in gc & pc:
            stats[c[0]][0] += 1
        for c in pc:
            stats[c[0]][1] += 1
        for c in gc:
            stats[c[0]][2] += 1
        tokens += len(g)
        correct_tokens += sum(a == b for a, b in zip(g, p))

    def prf(c, guessed, gold):
        p = 100 * c / guessed if guessed else 0.0
        r = 100 * c / gold if gold else 0.0
        return p, r, (2 * p * r / (p + r) if p + r else 0.0)

    total = [sum(v[i] for v in stats.values()) for i in range(3)]
    rows = {"overall": (*prf(*total), total[1], total[2])}
    for t in ["PER", "LOC", "ORG", "MISC"]:
        rows[t] = (*prf(*stats[t]), stats[t][1], stats[t][2])
    table = pd.DataFrame(rows, index=["precision", "recall", "F1", "predicted", "gold"]).T
    if verbose:
        print(f"processed {tokens} tokens with {total[2]} phrases; found: {total[1]} phrases; correct: {total[0]}.")
        print(f"accuracy: {100 * correct_tokens / tokens:6.2f}%; precision: {rows['overall'][0]:6.2f}%; "
              f"recall: {rows['overall'][1]:6.2f}%; FB1: {rows['overall'][2]:6.2f}")
        for t in ["PER", "LOC", "ORG", "MISC"]:
            p, r, f1, n_pred, _ = rows[t]
            print(f"{t:>17}: precision: {p:6.2f}%; recall: {r:6.2f}%; FB1: {f1:6.2f}  {int(n_pred)}")
    return table


pos_data = {k: make_dataset(v, "pos", POS2ID) for k, v in SPLITS.items()}
ner_data = {k: make_dataset(v, "ner", NER2ID) for k, v in SPLITS.items()}
ids_b, tags_b, len_b = next(batches(pos_data["train"], 4))
print("padded batch:", tuple(ids_b.shape), "| lengths:", len_b.tolist(), "| pad label positions:", int((tags_b == IGNORE).sum()))
print("conlleval chunk check:", get_chunks(["B-PER", "I-PER", "O", "I-LOC", "B-ORG", "B-ORG", "I-ORG"]))

In [ ]:
# linear-chain CRF: learnable transition matrix, forward algorithm for the NLL and Viterbi decoding
class CRF(nn.Module):
    def __init__(self, n_tags):
        super().__init__()
        self.n_tags = n_tags
        self.transitions = nn.Parameter(torch.zeros(n_tags, n_tags))   # transitions[i, j] = score of tag i -> tag j
        self.start = nn.Parameter(torch.zeros(n_tags))
        self.end = nn.Parameter(torch.zeros(n_tags))
        nn.init.uniform_(self.transitions, -0.1, 0.1)

    def gold_score(self, em, tags, mask):
        B, T, _ = em.shape
        ar = torch.arange(B, device=em.device)
        score = self.start[tags[:, 0]] + em[ar, 0, tags[:, 0]]
        for t in range(1, T):
            step = self.transitions[tags[:, t - 1], tags[:, t]] + em[ar, t, tags[:, t]]
            score = score + step * mask[:, t]
        last = tags.gather(1, (mask.sum(1).long() - 1).unsqueeze(1)).squeeze(1)
        return score + self.end[last]

    def log_partition(self, em, mask):
        alpha = self.start + em[:, 0]                                  # (B, n)
        for t in range(1, em.shape[1]):
            nxt = torch.logsumexp(alpha.unsqueeze(2) + self.transitions.unsqueeze(0) + em[:, t].unsqueeze(1), dim=1)
            alpha = torch.where(mask[:, t].unsqueeze(1).bool(), nxt, alpha)
        return torch.logsumexp(alpha + self.end, dim=1)

    def nll(self, em, tags, mask):
        tags = tags.masked_fill(~mask.bool(), 0)
        return self.log_partition(em, mask) - self.gold_score(em, tags, mask.float())

    @torch.no_grad()
    def viterbi(self, em, mask):
        B, T, n = em.shape
        score = self.start + em[:, 0]
        backpointers = []
        for t in range(1, T):
            total = score.unsqueeze(2) + self.transitions.unsqueeze(0)    # (B, from, to)
            best, idx = total.max(dim=1)
            nxt = best + em[:, t]
            m = mask[:, t].unsqueeze(1).bool()
            score = torch.where(m, nxt, score)
            backpointers.append(torch.where(m, idx, torch.arange(n, device=em.device).expand(B, n)))
        score = score + self.end
        paths = []
        lengths = mask.sum(1).long().tolist()
        for b in range(B):
            tag = int(score[b].argmax())
            path = [tag]
            for t in range(lengths[b] - 2, -1, -1):
                tag = int(backpointers[t][b, tag])
                path.append(tag)
            paths.append(path[::-1])
        return paths


# self-test on a tiny problem: Viterbi must equal the brute-force best path and the forward algorithm the brute-force log Z
import itertools
_g = torch.Generator().manual_seed(0)
_crf = CRF(3)
_em = torch.randn(1, 4, 3, generator=_g)
_mask = torch.ones(1, 4, dtype=torch.bool)
_scores = {seq: float(_crf.gold_score(_em, torch.tensor([seq]), _mask.float()).detach()) for seq in itertools.product(range(3), repeat=4)}
_best = max(_scores, key=_scores.get)
print("Viterbi path:", _crf.viterbi(_em, _mask)[0], "| brute-force best path:", list(_best))
print(f"forward-algorithm log Z = {float(_crf.log_partition(_em, _mask).detach()):.5f} | brute-force log Z = {float(torch.logsumexp(torch.tensor(list(_scores.values())), 0)):.5f}")
print("transition matrix is a learnable parameter:", _crf.transitions.requires_grad, tuple(_crf.transitions.shape))

In [ ]:
# the 2-layer BiLSTM tagger with either a softmax (linear + CE) or a CRF output layer
class BiLSTMTagger(nn.Module):
    def __init__(self, n_tags, emb_matrix=None, vocab_size=V, emb_dim=100, hidden=128, layers=2,
                 bidirectional=True, dropout=0.5, use_crf=False, freeze=False):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        if emb_matrix is not None:
            self.embedding.weight.data.copy_(torch.as_tensor(emb_matrix, dtype=torch.float32))
        self.embedding.weight.requires_grad = not freeze
        self.lstm = nn.LSTM(emb_dim, hidden, num_layers=layers, bidirectional=bidirectional, batch_first=True,
                            dropout=dropout if layers > 1 else 0.0)        # dropout between the LSTM layers
        self.out_dropout = nn.Dropout(dropout)
        self.emission = nn.Linear(hidden * (2 if bidirectional else 1), n_tags)
        self.crf = CRF(n_tags) if use_crf else None

    def contextual(self, ids, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(self.embedding(ids), lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=ids.shape[1])
        return out                                                     # h_t = [->h_t ; <-h_t], (B, T, 2*hidden)

    def forward(self, ids, lengths):
        return self.emission(self.out_dropout(self.contextual(ids, lengths)))

    def loss(self, ids, lengths, tags):
        em = self(ids, lengths)
        mask = tags != IGNORE
        if self.crf is not None:
            return self.crf.nll(em, tags, mask).mean()
        return F.cross_entropy(em.reshape(-1, em.shape[-1]), tags.reshape(-1), ignore_index=IGNORE)

    @torch.no_grad()
    def predict(self, ids, lengths):
        em = self(ids, lengths)
        mask = torch.arange(ids.shape[1]).unsqueeze(0) < lengths.unsqueeze(1)
        if self.crf is not None:
            return self.crf.viterbi(em, mask)
        return [row[:n].tolist() for row, n in zip(em.argmax(-1), lengths.tolist())]


probe = BiLSTMTagger(len(NER_TAGS), emb_c3, use_crf=True)
print("contextual representation shape:", tuple(probe.contextual(ids_b, len_b).shape),
      "| trainable params (fine-tuned):", sum(p.numel() for p in probe.parameters() if p.requires_grad))

In [ ]:
# training with Adam + weight decay, early stopping on validation F1 (patience 5), best checkpoint restored
def evaluate_tagger(model, data, tag_names, task):
    model.eval()
    gold, pred, losses = [], [], []
    with torch.no_grad():
        for ids, tags, lengths in batches(data, 32):
            losses.append(model.loss(ids, lengths, tags).item())
            for p, g, n in zip(model.predict(ids, lengths), tags, lengths.tolist()):
                gold.append([tag_names[i] for i in g[:n].tolist()])
                pred.append([tag_names[i] for i in p])
    flat_g = [t for s in gold for t in s]
    flat_p = [t for s in pred for t in s]
    if task == "pos":
        f1 = 100 * f1_score(flat_g, flat_p, average="macro", labels=sorted(set(flat_g) | set(flat_p)), zero_division=0)
    else:
        f1 = conlleval(gold, pred).loc["overall", "F1"]
    return {"loss": float(np.mean(losses)), "f1": float(f1), "acc": 100 * accuracy_score(flat_g, flat_p), "gold": gold, "pred": pred}


def train_tagger(task, data, cfg, seed=SEED, max_epochs=60, patience=5, verbose=True):
    set_seed(seed)
    tag_names = POS_TAGS if task == "pos" else NER_TAGS
    emb = None if cfg.get("random_init") else emb_c3
    model = BiLSTMTagger(len(tag_names), emb, bidirectional=cfg.get("bidirectional", True),
                         dropout=cfg.get("dropout", 0.5), use_crf=cfg.get("crf", False), freeze=cfg.get("freeze", False))
    opt = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=1e-4, fused=True)
    rng = random.Random(seed)
    history, best, best_state, wait, t0 = [], (-1.0, 0.0), None, 0, time.time()
    for epoch in range(1, max_epochs + 1):
        model.train()
        train_losses = []
        for ids, tags, lengths in batches(data["train"], 16, shuffle=True, rng=rng):
            loss = model.loss(ids, lengths, tags)
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            train_losses.append(loss.item())
        val = evaluate_tagger(model, data["val"], tag_names, task)
        history.append({"epoch": epoch, "train_loss": float(np.mean(train_losses)), "val_loss": val["loss"], "val_f1": val["f1"]})
        if (val["f1"], -val["loss"]) > best:       # higher val F1, or equal F1 with lower val loss (F1 is flat at 0 early on)
            best, best_state, wait = (val["f1"], -val["loss"]), copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    best = best[0]
    secs = time.time() - t0
    if verbose:
        print(f"{task.upper():3s} {cfg.get('name', ''):28s} epochs run {epoch:2d} | best epoch {int(np.argmax([h['val_f1'] for h in history])) + 1:2d} "
              f"| best val F1 {best:5.2f} | {secs / epoch:.2f}s/epoch")
    return model, pd.DataFrame(history), {"best_val_f1": best, "epochs": epoch, "sec_per_epoch": secs / epoch}


MAIN_CONFIGS = {
    ("pos", "frozen"): {"name": "frozen W2V, softmax", "freeze": True},
    ("pos", "fine-tuned"): {"name": "fine-tuned W2V, softmax", "freeze": False},
    ("ner", "frozen"): {"name": "frozen W2V, CRF", "freeze": True, "crf": True},
    ("ner", "fine-tuned"): {"name": "fine-tuned W2V, CRF", "freeze": False, "crf": True},
}
main_runs = {}
for (task, mode), cfg in MAIN_CONFIGS.items():
    main_runs[(task, mode)] = train_tagger(task, pos_data if task == "pos" else ner_data, cfg)

In [ ]:
# training / validation loss curves per epoch for the four main runs
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, ((task, mode), (_, hist, info)) in zip(axes.ravel(), main_runs.items()):
    ax.plot(hist["epoch"], hist["train_loss"], marker="o", ms=3, label="train loss")
    ax.plot(hist["epoch"], hist["val_loss"], marker="o", ms=3, label="validation loss")
    best_epoch = int(hist["val_f1"].idxmax()) + 1
    ax.axvline(best_epoch, color="grey", ls="--", lw=1, label=f"best val F1 (epoch {best_epoch})")
    ax.set_title(f"{task.upper()} BiLSTM, {mode} embeddings" + (" + CRF" if task == "ner" else ""))
    ax.set_xlabel("epoch")
    ax.set_ylabel("CRF negative log-likelihood" if task == "ner" else "cross-entropy loss")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
plt.tight_layout()
savefig("bilstm_curves.png")
plt.show()

In [ ]:
# frozen vs fine-tuned summary (validation and test), and saving the selected models
summary_rows = []
for (task, mode), (model, hist, info) in main_runs.items():
    names = POS_TAGS if task == "pos" else NER_TAGS
    test = evaluate_tagger(model, (pos_data if task == "pos" else ner_data)["test"], names, task)
    summary_rows.append({"task": task.upper(), "embeddings": mode, "epochs run": info["epochs"],
                         "best val F1": round(info["best_val_f1"], 2), "test token acc": round(test["acc"], 2),
                         "test F1": round(test["f1"], 2)})
mode_table = pd.DataFrame(summary_rows)
display(mode_table)
best_mode = {t: max(["frozen", "fine-tuned"], key=lambda m: main_runs[(t, m)][2]["best_val_f1"]) for t in ("pos", "ner")}
print("selected on validation F1:", best_mode)
pos_model, ner_model = main_runs[("pos", best_mode["pos"])][0], main_runs[("ner", best_mode["ner"])][0]
torch.save({"state_dict": pos_model.state_dict(), "tags": POS_TAGS, "mode": best_mode["pos"], "config": MAIN_CONFIGS[("pos", best_mode["pos"])]}, "models/bilstm_pos.pt")
torch.save({"state_dict": ner_model.state_dict(), "tags": NER_TAGS, "mode": best_mode["ner"], "config": MAIN_CONFIGS[("ner", best_mode["ner"])]}, "models/bilstm_ner.pt")
print("saved models/bilstm_pos.pt and models/bilstm_ner.pt")

## 5. Evaluation
### 5.1 POS tagging (test set)

In [ ]:
# token accuracy, macro-F1 and the confusion matrix over all POS tags
pos_test = evaluate_tagger(pos_model, pos_data["test"], POS_TAGS, "pos")
flat_g = [t for s in pos_test["gold"] for t in s]
flat_p = [t for s in pos_test["pred"] for t in s]
print(f"POS test ({best_mode['pos']} embeddings): token accuracy = {pos_test['acc']:.2f}% | macro-F1 = {pos_test['f1']:.2f}")
per_tag = f1_score(flat_g, flat_p, labels=POS_TAGS, average=None, zero_division=0)
display(pd.DataFrame({"support": [flat_g.count(t) for t in POS_TAGS], "F1": (100 * per_tag).round(1)}, index=POS_TAGS).T)

cm = confusion_matrix(flat_g, flat_p, labels=POS_TAGS)
fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(np.log1p(cm), cmap="Blues")
for i in range(len(POS_TAGS)):
    for j in range(len(POS_TAGS)):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8, color="white" if cm[i, j] > cm.max() / 4 else "black")
ax.set_xticks(range(len(POS_TAGS)), POS_TAGS, rotation=45)
ax.set_yticks(range(len(POS_TAGS)), POS_TAGS)
ax.set_xlabel("predicted tag")
ax.set_ylabel("gold (silver) tag")
ax.set_title("POS confusion matrix on the test set (counts, colour = log scale)")
plt.colorbar(im, ax=ax, label="log(1 + count)")
savefig("pos_confusion.png")
plt.show()

In [ ]:
# the 3 most confused tag pairs with example sentences (test set first, then validation if needed)
pairs = Counter()
for g, p in zip(flat_g, flat_p):
    if g != p:
        pairs[tuple(sorted((g, p)))] += 1
top_pairs = [pr for pr, _ in pairs.most_common(3)]
print("most confused pairs (both directions counted):", pairs.most_common(3))
pos_val_eval = evaluate_tagger(pos_model, pos_data["val"], POS_TAGS, "pos")
for a, b in top_pairs:
    shown = 0
    print(f"\n=== {a} <-> {b} ===")
    for split_name, ev, items in (("test", pos_test, SPLITS["test"]), ("val", pos_val_eval, SPLITS["val"])):
        for x, g, p in zip(items, ev["gold"], ev["pred"]):
            hits = [k for k in range(len(g)) if {g[k], p[k]} == {a, b}]
            if hits and shown < 2:
                marked = " ".join(f"[{t} gold={g[k]} pred={p[k]}]" if k in hits else t for k, t in enumerate(x["tokens"]))
                print(f"({split_name}) {marked}")
                shown += 1

DISC_POS

### 5.2 NER (test set, conlleval)

In [ ]:
# entity-level P / R / F1 per type with the CRF model, and the same network with a softmax output layer
ner_test = evaluate_tagger(ner_model, ner_data["test"], NER_TAGS, "ner")
print(f"BiLSTM + CRF ({best_mode['ner']} embeddings):")
crf_table = conlleval(ner_test["gold"], ner_test["pred"], verbose=True)
softmax_ner_model, softmax_hist, softmax_info = train_tagger("ner", ner_data, {"name": f"{best_mode['ner']} W2V, softmax (no CRF)",
                                                                             "freeze": best_mode["ner"] == "frozen", "crf": False})
ner_test_soft = evaluate_tagger(softmax_ner_model, ner_data["test"], NER_TAGS, "ner")
print("\nBiLSTM + softmax (no CRF):")
soft_table = conlleval(ner_test_soft["gold"], ner_test_soft["pred"], verbose=True)
display(pd.concat({"with CRF": crf_table[["precision", "recall", "F1"]], "without CRF": soft_table[["precision", "recall", "F1"]]}, axis=1).round(2))


def invalid_transitions(seqs):
    return sum(1 for s in seqs for prev, cur in zip(["O"] + s, s) if cur.startswith("I-") and prev[2:] != cur[2:])


print("invalid BIO transitions (I-X not after B-X/I-X) in test predictions: CRF =", invalid_transitions(ner_test["pred"]),
      "| softmax =", invalid_transitions(ner_test_soft["pred"]))

In [ ]:
# error analysis: 5 false positives and 5 false negatives of the CRF model, each with an automatic diagnosis
train_entity_strings = {" ".join(x["tokens"][s:e + 1]) for x in SPLITS["train"] for _, s, e in get_chunks(x["ner"])}


def diagnose(kind, chunk, other_chunks, tokens):
    etype, s, e = chunk
    text = " ".join(tokens[s:e + 1])
    for ot, os_, oe in other_chunks:
        if (os_, oe) == (s, e) and ot != etype:
            return f"type confusion: same span is {ot} in the {'gold' if kind == 'FP' else 'prediction'}"
        if os_ <= e and s <= oe:
            return f"boundary error: overlaps {'gold' if kind == 'FP' else 'predicted'} {ot} span '{' '.join(tokens[os_:oe + 1])}'"
    if kind == "FN":
        return "missed entity, " + ("string seen in training" if text in train_entity_strings else "never seen as an entity in training (needs a context cue the model did not learn)")
    if text in train_entity_strings:
        return "spurious entity: this string is an entity elsewhere in training, the model ignored the context"
    toks = tokens[s:e + 1]
    if all(t in LEX.FIRST_NAMES | LEX.SURNAMES for t in toks) or (s > 0 and tokens[s - 1] in LEX.FIRST_NAMES | LEX.PERSON_TITLES):
        return "likely a real name the silver rules missed (name part / title context); a label-noise 'error'"
    if all(t in LEX.LETTER_NAMES for t in toks):
        return "acronym letters: the model generalised the ORG acronym pattern (e.g. پی ٹی آئی) to another acronym"
    return "spurious entity: name-like context (person name or postposition pattern) triggered a false span"


fps, fns = [], []
for x, g, p in zip(SPLITS["test"], ner_test["gold"], ner_test["pred"]):
    gc, pc = set(get_chunks(g)), set(get_chunks(p))
    fps += [(x["tokens"], c, gc) for c in pc - gc]
    fns += [(x["tokens"], c, pc) for c in gc - pc]
print(f"test errors: {len(fps)} false positives, {len(fns)} false negatives")
err_rows = []
for kind, errors in (("FP", fps[:5]), ("FN", fns[:5])):
    for tokens, chunk, others in errors:
        etype, s, e = chunk
        ctx = " ".join(tokens[max(0, s - 4):e + 5])
        err_rows.append({"kind": kind, "type": etype, "span": " ".join(tokens[s:e + 1]), "context": ctx,
                         "explanation": diagnose(kind, chunk, others, tokens)})
display(pd.DataFrame(err_rows))

DISC_NER

### 5.3 Ablation study
Every ablation changes exactly one thing relative to the selected baseline (2-layer BiLSTM, C3 embeddings in
the mode chosen on validation F1 in Section 4, dropout 0.5; softmax head for POS, CRF head for NER) and is trained on the same split. Because the
test set is small (75 sentences), each configuration is trained with 3 seeds and the mean ± std of the test
score is reported (POS: macro-F1 and accuracy; NER: conlleval entity F1).

In [ ]:
# A1 unidirectional, A2 no dropout, A3 random embedding init, A4 softmax instead of CRF (NER only); 3 seeds each
ABLATION_SEEDS = [SEED, SEED + 1, SEED + 2]
base = {"freeze": False, "dropout": 0.5, "bidirectional": True}
ABLATIONS = {
    "baseline (selected BiLSTM)": {},
    "A1 unidirectional LSTM": {"bidirectional": False},
    "A2 no dropout": {"dropout": 0.0},
    "A3 random embedding init": {"random_init": True},
    "A4 softmax instead of CRF": {"crf": False},
}
abl_rows = []
for task in ("pos", "ner"):
    data = pos_data if task == "pos" else ner_data
    names = POS_TAGS if task == "pos" else NER_TAGS
    for abl_name, change in ABLATIONS.items():
        if task == "pos" and abl_name.startswith("A4"):
            continue
        scores = []
        for seed in ABLATION_SEEDS:
            cfg = {**base, "freeze": best_mode[task] == "frozen", "crf": task == "ner", **change, "name": f"{abl_name} s{seed}"}
            if cfg.get("random_init"):
                cfg["freeze"] = False                                # random vectors must be trainable
            model, hist, info = train_tagger(task, data, cfg, seed=seed, verbose=False)
            test = evaluate_tagger(model, data["test"], names, task)
            scores.append((test["f1"], test["acc"], info["epochs"]))
        f1s, accs, eps = map(np.array, zip(*scores))
        abl_rows.append({"task": task.upper(), "configuration": abl_name, "test F1 mean": f1s.mean().round(2),
                         "test F1 std": f1s.std().round(2), "token acc mean": accs.mean().round(2), "epochs (mean)": eps.mean().round(1)})
        print(f"{task.upper()} {abl_name:36s} F1 = {f1s.mean():5.2f} ± {f1s.std():4.2f} | acc = {accs.mean():5.2f}")
ablation_table = pd.DataFrame(abl_rows)
for task in ("POS", "NER"):
    sub = ablation_table[ablation_table.task == task].copy()
    sub["delta F1 vs baseline"] = (sub["test F1 mean"] - sub["test F1 mean"].iloc[0]).round(2)
    display(sub.set_index("configuration"))

DISC_ABL

# Part 3: Transformer Encoder for Topic Classification
## 6. Dataset preparation
Labels are the five categories assigned in Part 0 (keyword guided). Each article becomes the token-id sequence of
its `cleaned.txt` sentences (shared 10,002-word `word2idx`), truncated or padded to 256 tokens; a learned `[CLS]`
vector is prepended inside the model, so the encoder sees 257 positions.

In [ ]:
# token-id sequences (256 tokens), 70/15/15 split stratified by category, class distribution
MAX_TOKENS = 256
article_ids_arr = np.array(ARTICLE_IDS)
article_labels = np.array([CAT2ID[article_topic[n]] for n in ARTICLE_IDS])
seq_matrix = np.zeros((len(ARTICLE_IDS), MAX_TOKENS), dtype=np.int64)
true_lengths = []
for r, num in enumerate(ARTICLE_IDS):
    ids = encode([t for s in clean_docs[num] for t in s])
    true_lengths.append(len(ids))
    seq_matrix[r, :min(MAX_TOKENS, len(ids))] = ids[:MAX_TOKENS]
print(f"article length in tokens: median {int(np.median(true_lengths))}, min {min(true_lengths)}, max {max(true_lengths)} "
      f"| truncated to 256: {sum(l > MAX_TOKENS for l in true_lengths)} of {len(true_lengths)}")

tr_idx, rest_idx = train_test_split(np.arange(len(ARTICLE_IDS)), test_size=0.30, stratify=article_labels, random_state=SEED)
va_idx, te_idx = train_test_split(rest_idx, test_size=0.50, stratify=article_labels[rest_idx], random_state=SEED)
CLS_SPLITS = {"train": tr_idx, "val": va_idx, "test": te_idx}
dist = pd.DataFrame({k: Counter(article_labels[v]) for k, v in CLS_SPLITS.items()}).sort_index()
dist.index = CATEGORIES
dist.loc["total"] = dist.sum()
display(dist)


def cls_batches(idx, batch_size=16, shuffle=False, rng=None):
    idx = np.array(idx)
    if shuffle:
        idx = idx[rng.permutation(len(idx))]
    for i in range(0, len(idx), batch_size):
        b = idx[i:i + batch_size]
        yield torch.from_numpy(seq_matrix[b]), torch.from_numpy(article_labels[b])

## 7. Transformer encoder (every component is its own `nn.Module`, no PyTorch Transformer classes)

In [ ]:
# scaled dot-product attention with an optional padding mask; returns the output and the attention weights
class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        # q, k: (B, T, d_k), v: (B, T, d_v), mask: (B, 1, T) with True = real token
        scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        return self.dropout(weights) @ v, weights


_q = torch.randn(2, 5, 32)
_mask = torch.tensor([[1, 1, 1, 0, 0], [1, 1, 1, 1, 1]], dtype=torch.bool).unsqueeze(1)
_out, _w = ScaledDotProductAttention(0.0)(_q, _q, _q, _mask)
print("output:", tuple(_out.shape), "| weights:", tuple(_w.shape), "| rows sum to 1:", torch.allclose(_w.sum(-1), torch.ones(2, 5)),
      "| weight on masked keys:", float(_w[0, :, 3:].sum()))

In [ ]:
# multi-head self-attention: h = 4 heads, each with its own W_q, W_k, W_v (d_model -> 32), shared output projection
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model=128, n_heads=4, d_k=32, d_v=32, dropout=0.1):
        super().__init__()
        self.W_q = nn.ModuleList([nn.Linear(d_model, d_k, bias=False) for _ in range(n_heads)])
        self.W_k = nn.ModuleList([nn.Linear(d_model, d_k, bias=False) for _ in range(n_heads)])
        self.W_v = nn.ModuleList([nn.Linear(d_model, d_v, bias=False) for _ in range(n_heads)])
        self.W_o = nn.Linear(n_heads * d_v, d_model)
        self.attention = ScaledDotProductAttention(dropout)

    def forward(self, x, mask=None):
        outputs, weights = [], []
        for W_q, W_k, W_v in zip(self.W_q, self.W_k, self.W_v):
            out, w = self.attention(W_q(x), W_k(x), W_v(x), mask)
            outputs.append(out)
            weights.append(w)
        return self.W_o(torch.cat(outputs, dim=-1)), torch.stack(weights, dim=1)   # (B, T, d_model), (B, h, T, T)


_mha = MultiHeadSelfAttention()
_o, _w = _mha(torch.randn(2, 7, 128))
print("MHA output:", tuple(_o.shape), "| attention weights:", tuple(_w.shape), "| heads with own W_q/W_k/W_v:", len(_mha.W_q),
      "| per-head projection:", tuple(_mha.W_q[0].weight.shape), "| shared W_o:", tuple(_mha.W_o.weight.shape))

In [ ]:
# position-wise feed-forward network: 128 -> 512 -> 128 with ReLU (and dropout)
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model=128, d_ff=512, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model))

    def forward(self, x):
        return self.net(x)


_ffn = PositionwiseFeedForward()
print("FFN:", _ffn.net, "| output shape:", tuple(_ffn(torch.randn(2, 7, 128)).shape))

In [ ]:
# sinusoidal positional encoding stored as a fixed (non-learned) buffer and added to the input embeddings
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model=128, max_len=512):
        super().__init__()
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.pow(10000.0, torch.arange(0, d_model, 2).float() / d_model)
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos / div)                    # PE(pos, 2i)   = sin(pos / 10000^(2i/d))
        pe[:, 1::2] = torch.cos(pos / div)                    # PE(pos, 2i+1) = cos(pos / 10000^(2i/d))
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[: x.shape[1]].unsqueeze(0)


_pe = SinusoidalPositionalEncoding()
print("PE buffer:", tuple(_pe.pe.shape), "| registered buffers:", [n for n, _ in _pe.named_buffers()],
      "| learnable parameters:", sum(p.numel() for p in _pe.parameters()), "| PE(0) = [sin 0, cos 0, ...] =", _pe.pe[0, :4].tolist())

In [ ]:
# Pre-LN encoder block: x <- x + Dropout(MHA(LN(x))); x <- x + Dropout(FFN(LN(x)))
class EncoderBlock(nn.Module):
    def __init__(self, d_model=128, n_heads=4, d_k=32, d_v=32, d_ff=512, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.mha = MultiHeadSelfAttention(d_model, n_heads, d_k, d_v, dropout)
        self.drop1 = nn.Dropout(dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, weights = self.mha(self.ln1(x), mask)
        x = x + self.drop1(attn_out)
        x = x + self.drop2(self.ffn(self.ln2(x)))
        return x, weights


_blk = EncoderBlock()
_y, _w = _blk(torch.randn(2, 7, 128))
print("encoder block output:", tuple(_y.shape), "| attention:", tuple(_w.shape), "| Pre-LN modules:", [n for n, _ in _blk.named_children()])

In [ ]:
# full classifier: token embedding + [CLS] + positional encoding -> 4 Pre-LN blocks -> MLP 128 -> 64 -> 5 on [CLS]
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size=V, n_classes=5, d_model=128, n_heads=4, d_k=32, d_v=32, d_ff=512, n_layers=4, dropout=0.1, max_len=512):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)          # learned [CLS] token
        self.positional = SinusoidalPositionalEncoding(d_model, max_len)
        self.input_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([EncoderBlock(d_model, n_heads, d_k, d_v, d_ff, dropout) for _ in range(n_layers)])
        self.final_ln = nn.LayerNorm(d_model)
        self.head = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, n_classes))

    def forward(self, ids, return_attention=False):
        B = ids.shape[0]
        x = self.embedding(ids) * math.sqrt(self.d_model)
        x = torch.cat([self.cls.expand(B, -1, -1), x], dim=1)                # prepend [CLS]
        mask = torch.cat([torch.ones(B, 1, dtype=torch.bool, device=ids.device), ids != 0], dim=1).unsqueeze(1)  # (B, 1, T+1)
        x = self.input_dropout(self.positional(x))
        attentions = []
        for block in self.blocks:
            x, w = block(x, mask)
            attentions.append(w)
        logits = self.head(self.final_ln(x[:, 0]))
        return (logits, attentions) if return_attention else logits


probe_tf = TransformerClassifier()
with torch.no_grad():
    logits, atts = probe_tf(torch.from_numpy(seq_matrix[:2]), return_attention=True)
print("logits:", tuple(logits.shape), "| attention per layer:", tuple(atts[-1].shape), "| layers:", len(atts),
      "| parameters:", sum(p.numel() for p in probe_tf.parameters()),
      "| PE is a buffer, not a parameter:", "positional.pe" in dict(probe_tf.named_buffers()) and not any("pe" in n for n, _ in probe_tf.named_parameters()))
print("rows of the attention matrix sum to 1:", torch.allclose(atts[-1].sum(-1), torch.ones_like(atts[-1].sum(-1))),
      "| attention on padding positions:", float(atts[-1][1, :, :, 1:][:, :, torch.from_numpy(seq_matrix[1] == 0)].sum()))

In [ ]:
# training: AdamW (5e-4, weight decay 0.01), cosine schedule with 50 linear warm-up steps, 20 epochs, batch 16
def cosine_with_warmup(optimizer, warmup, total):
    def factor(step):
        if step < warmup:
            return (step + 1) / warmup
        return 0.5 * (1.0 + math.cos(math.pi * (step - warmup) / max(1, total - warmup)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)


def run_classifier_epoch(model, idx, opt=None, sched=None, rng=None):
    training = opt is not None
    model.train(training)
    losses, correct, n = [], 0, 0
    with torch.set_grad_enabled(training):
        for ids, y in cls_batches(idx, 16, shuffle=training, rng=rng):
            logits = model(ids)
            loss = F.cross_entropy(logits, y)
            if training:
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                if sched is not None:
                    sched.step()
            losses.append(loss.item() * len(y))
            correct += int((logits.argmax(-1) == y).sum())
            n += len(y)
    return sum(losses) / n, 100 * correct / n


def train_classifier(model, opt, sched=None, epochs=20, tag=""):
    rng = np.random.default_rng(SEED)
    hist, best, best_state, lrs = [], (-1, 0), None, []
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        e0 = time.time()
        tr_loss, tr_acc = run_classifier_epoch(model, tr_idx, opt, sched, rng)
        secs = time.time() - e0
        va_loss, va_acc = run_classifier_epoch(model, va_idx)
        lrs.append(opt.param_groups[0]["lr"])
        hist.append({"epoch": epoch, "train_loss": tr_loss, "train_acc": tr_acc, "val_loss": va_loss, "val_acc": va_acc, "sec": secs})
        if (va_acc, -va_loss) > best:
            best, best_state = (va_acc, -va_loss), copy.deepcopy(model.state_dict())
        print(f"[{tag}] epoch {epoch:2d} | train loss {tr_loss:.3f} acc {tr_acc:5.1f} | val loss {va_loss:.3f} acc {va_acc:5.1f} | lr {lrs[-1]:.2e} | {secs:.1f}s")
    model.load_state_dict(best_state)
    return pd.DataFrame(hist), lrs


set_seed()
transformer = TransformerClassifier()
tf_opt = torch.optim.AdamW(transformer.parameters(), lr=5e-4, weight_decay=0.01)
steps_per_epoch = math.ceil(len(tr_idx) / 16)
tf_sched = cosine_with_warmup(tf_opt, warmup=50, total=20 * steps_per_epoch)
print(f"steps per epoch: {steps_per_epoch} | total steps: {20 * steps_per_epoch} | warm-up steps: 50")
tf_hist, tf_lrs = train_classifier(transformer, tf_opt, tf_sched, epochs=20, tag="Transformer")
torch.save({"state_dict": transformer.state_dict(), "categories": CATEGORIES, "max_tokens": MAX_TOKENS}, "models/transformer_cls.pt")
print("saved models/transformer_cls.pt (best validation epoch:", int(tf_hist.sort_values(["val_acc", "val_loss"], ascending=[False, True]).iloc[0]["epoch"]), ")")

In [ ]:
# training / validation loss and accuracy per epoch (plus the learning-rate schedule)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
axes[0].plot(tf_hist.epoch, tf_hist.train_loss, marker="o", ms=3, label="train")
axes[0].plot(tf_hist.epoch, tf_hist.val_loss, marker="o", ms=3, label="validation")
axes[0].set(title="Transformer: cross-entropy loss per epoch", xlabel="epoch", ylabel="loss")
axes[1].plot(tf_hist.epoch, tf_hist.train_acc, marker="o", ms=3, label="train")
axes[1].plot(tf_hist.epoch, tf_hist.val_acc, marker="o", ms=3, label="validation")
axes[1].set(title="Transformer: accuracy per epoch", xlabel="epoch", ylabel="accuracy (%)")
axes[2].plot(range(1, 21), tf_lrs, marker="o", ms=3, color="#8c564b")
axes[2].set(title="learning rate at the end of each epoch (warm-up 50 steps + cosine)", xlabel="epoch", ylabel="learning rate")
for ax in axes[:2]:
    ax.legend()
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
savefig("transformer_curves.png")
plt.show()

## 8. Evaluation
### 8.1 Results

In [ ]:
# test accuracy, macro-F1 and the 5 x 5 confusion matrix
def predict_classifier(model, idx):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(ids).argmax(-1) for ids, _ in cls_batches(idx, 32)]).numpy()


tf_pred = predict_classifier(transformer, te_idx)
y_test = article_labels[te_idx]
tf_acc, tf_f1 = 100 * accuracy_score(y_test, tf_pred), 100 * f1_score(y_test, tf_pred, average="macro")
print(f"Transformer test accuracy = {tf_acc:.2f}% | macro-F1 = {tf_f1:.2f} (n = {len(te_idx)} articles)")
cm5 = confusion_matrix(y_test, tf_pred, labels=range(5))
fig, ax = plt.subplots(figsize=(6.5, 5.2))
im = ax.imshow(cm5, cmap="Blues")
for i in range(5):
    for j in range(5):
        ax.text(j, i, cm5[i, j], ha="center", va="center", color="white" if cm5[i, j] > cm5.max() / 2 else "black")
ax.set_xticks(range(5), CATEGORIES, rotation=30)
ax.set_yticks(range(5), CATEGORIES)
ax.set_xlabel("predicted category")
ax.set_ylabel("true category")
ax.set_title("Transformer: 5 x 5 confusion matrix (test set)")
plt.colorbar(im, ax=ax, label="articles")
savefig("transformer_confusion.png")
plt.show()

In [ ]:
# attention heatmaps: final encoder layer, heads 1 and 2, for 3 correctly classified test articles
# (257 x 257 is unreadable, so each map shows [CLS] plus the 23 tokens [CLS] attends to most, kept in sentence order)
SHOW = 24
correct_pos = [i for i, (p, y) in enumerate(zip(tf_pred, y_test)) if p == y]
chosen, seen_cls = [], set()
for i in correct_pos:                                       # prefer three different categories
    if y_test[i] not in seen_cls:
        chosen.append(i)
        seen_cls.add(y_test[i])
    if len(chosen) == 3:
        break
chosen += [i for i in correct_pos if i not in chosen][: 3 - len(chosen)]
transformer.eval()
fig, axes = plt.subplots(3, 2, figsize=(14, 19))
cls_focus = []
for row, i in enumerate(chosen):
    r = te_idx[i]
    with torch.no_grad():
        _, atts = transformer(torch.from_numpy(seq_matrix[r:r + 1]), return_attention=True)
    final = atts[-1][0]                                           # (heads, 257, 257)
    n_real = int((seq_matrix[r] != 0).sum())
    cls_mean = final[:, 0, 1:n_real + 1].mean(0)
    keep = [0] + sorted((torch.argsort(cls_mean, descending=True)[:SHOW - 1] + 1).tolist())   # positions incl. [CLS]
    labels = ["[CLS]"] + [ur(idx2word[seq_matrix[r, p - 1]]) for p in keep[1:]]
    for col, head in enumerate((0, 1)):
        ax = axes[row, col]
        im = ax.imshow(final[head][keep][:, keep].numpy(), cmap="viridis")
        ax.set_xticks(range(SHOW), labels, rotation=90, fontsize=9)
        ax.set_yticks(range(SHOW), labels, fontsize=9)
        ax.set_xlabel("key (attended-to token)")
        ax.set_ylabel("query token")
        ax.set_title(f"article {ARTICLE_IDS[r]} ({CATEGORIES[y_test[i]]}), final layer, head {head + 1}")
        plt.colorbar(im, ax=ax, fraction=0.046)
    cls_row = cls_mean.numpy()          # [CLS] -> tokens, averaged over heads
    top = np.argsort(-cls_row)[:8]
    cls_focus.append({"article": ARTICLE_IDS[r], "category": CATEGORIES[y_test[i]],
                      "tokens [CLS] attends to most (final layer, mean of heads)": " | ".join(f"{idx2word[seq_matrix[r, t]]} ({cls_row[t]:.3f})" for t in top)})
plt.tight_layout()
savefig("attention_heatmaps.png")
plt.show()
display(pd.DataFrame(cls_focus))

DISC_TF

### 8.2 BiLSTM vs Transformer comparison
The Part 2 BiLSTM is a token tagger, so to compare the two architectures on the same task the BiLSTM encoder from
Part 2 (2 layers, bidirectional, hidden 128, dropout 0.5, fine-tuned C3 embeddings) is given a classification
head (masked mean + max pooling -> 64 -> 5) and trained on exactly the same split, sequences and number of epochs.

In [ ]:
# BiLSTM classifier on the same 256-token sequences, 20 epochs, Adam (1e-3, wd 1e-4)
class BiLSTMClassifier(nn.Module):
    def __init__(self, emb_matrix, n_classes=5, hidden=128, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(torch.as_tensor(emb_matrix, dtype=torch.float32), freeze=False, padding_idx=0)
        self.lstm = nn.LSTM(emb_matrix.shape[1], hidden, num_layers=2, bidirectional=True, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Sequential(nn.Linear(4 * hidden, 64), nn.ReLU(), nn.Linear(64, n_classes))

    def forward(self, ids):
        lengths = (ids != 0).sum(1).clamp(min=1)
        packed = nn.utils.rnn.pack_padded_sequence(self.embedding(ids), lengths, batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=ids.shape[1])
        mask = (ids != 0).unsqueeze(-1)
        mean = (out * mask).sum(1) / lengths.unsqueeze(1)
        mx = out.masked_fill(~mask, -1e9).max(1).values
        return self.head(self.dropout(torch.cat([mean, mx], dim=-1)))


set_seed()
bilstm_cls = BiLSTMClassifier(emb_c3)
lstm_opt = torch.optim.Adam(bilstm_cls.parameters(), lr=1e-3, weight_decay=1e-4)
lstm_hist, _ = train_classifier(bilstm_cls, lstm_opt, None, epochs=20, tag="BiLSTM")
lstm_pred = predict_classifier(bilstm_cls, te_idx)
lstm_acc, lstm_f1 = 100 * accuracy_score(y_test, lstm_pred), 100 * f1_score(y_test, lstm_pred, average="macro")

In [ ]:
# side-by-side numbers used in the written comparison
def converged_epoch(hist):
    """first epoch whose validation accuracy reaches 95% of the best validation accuracy of the run"""
    return int(hist.loc[hist.val_acc >= 0.95 * hist.val_acc.max(), "epoch"].iloc[0])


comparison = pd.DataFrame({
    "Transformer (from scratch)": [tf_acc, tf_f1, tf_hist.val_acc.max(), int(tf_hist.val_acc.idxmax()) + 1, converged_epoch(tf_hist),
                                   tf_hist.sec.mean(), sum(p.numel() for p in transformer.parameters())],
    "BiLSTM (C3 init)": [lstm_acc, lstm_f1, lstm_hist.val_acc.max(), int(lstm_hist.val_acc.idxmax()) + 1, converged_epoch(lstm_hist),
                         lstm_hist.sec.mean(), sum(p.numel() for p in bilstm_cls.parameters())],
}, index=["test accuracy (%)", "test macro-F1", "best val accuracy (%)", "best val epoch", "epoch reaching 95% of best val acc",
          "train seconds / epoch", "parameters"]).round(2)
display(comparison)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
for name, h in (("Transformer", tf_hist), ("BiLSTM", lstm_hist)):
    axes[0].plot(h.epoch, h.val_acc, marker="o", ms=3, label=name)
    axes[1].plot(h.epoch, h.train_loss, marker="o", ms=3, label=name)
axes[0].set(title="validation accuracy per epoch", xlabel="epoch", ylabel="accuracy (%)")
axes[1].set(title="training loss per epoch", xlabel="epoch", ylabel="cross-entropy")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
savefig("bilstm_vs_transformer.png")
plt.show()
cm_lstm = confusion_matrix(y_test, lstm_pred, labels=range(5))
display(pd.DataFrame(cm_lstm, index=[f"true {c}" for c in CATEGORIES], columns=[f"pred {c}" for c in CATEGORIES]))

DISC_COMPARE

DISC_CONCLUSION